# Position monitor — does the reason you entered still hold?

Reads your currently open F&O **option** positions directly from Kite — no manual
ledger. What you're actually holding is inferred from the legs themselves: a
net-short **CE** implies a **bear call** (thesis = DOWN), a net-short **PE** implies
a **bull put** (thesis = UP). That inferred thesis is then checked against
`daily_screener.check_trend_status()` for the same underlying — the exact same
tested logic the entry screener (`06_daily_screener.ipynb`) already uses to build
the tradeable list. Nothing new is invented here; this just re-points that logic at
positions instead of candidates.

**Deliberately does NOT do:** propose a profit-target or DTE-based exit. Those
have never been tested against this data (see README, `exhaustion_test.py`
finding) and stay out of scope. This only answers *has the entry thesis changed*,
using columns already validated elsewhere in this project.

| Column | Meaning |
|---|---|
| `structure` | Inferred from your option legs — BEAR CALL / BULL PUT / mixed |
| `live_pnl` | Real unrealised P&L from Kite, not modelled — this project has no IV/option-pricing data |
| `current_direction` | The Supertrend flip direction **right now**, freshly refreshed |
| `thesis_status` | `thesis intact` · `⚠ FLIP REVERSED` — the direction you're structured for is no longer open · `⚠ signal weakened` — this name has slid into watchlist/no_edge since you likely entered · `⚠ buffer to flip now X%` — reusing the same 1.5% cutoff the entry screener itself applies, not a new threshold |
| `dte` | Days to this leg's expiry, for context only |

**Still entirely manual** — this reports state changes, it does not tell you to
exit or enter anything. Read `thesis_status` as "here's what changed since you
likely opened this," same evidentiary standard as the rest of the project.

In [1]:
# Which underlyings are actually held right now, straight from Kite — before
# refreshing anything. Cheap, no need to touch the historical pipeline just to
# find out what to refresh.

%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

from src.auth import get_kite
from src.position_monitor import get_open_option_positions

kite = get_kite()
legs = get_open_option_positions(kite)

if legs.empty:
    held_symbols = []
    print("No open option positions found.")
else:
    held_symbols = sorted(legs["name"].unique().tolist())
    print(f"Held via options: {held_symbols}")
    legs[["tradingsymbol", "name", "instrument_type", "strike", "expiry", "quantity", "pnl"]]

Held via options: ['DABUR', 'PIIND', 'TVSMOTOR']


In [2]:
# Refresh ONLY the held underlyings — same incremental fetch used by the daily
# screener (catch-up window merged onto saved history, not a full re-pull), so
# the thesis check below runs against today's flip state, not a stale one.
# Indices need get_index_token(); everything else uses the NSE token lookup.

from src.instruments import build_nse_token_lookup, get_index_token
from src.scale_pipeline import process_stock_incremental
from datetime import datetime

index_symbols = {"NIFTY": ("NIFTY 50", "NSE"), "BANKNIFTY": ("NIFTY BANK", "NSE"),
                  "FINNIFTY": ("NIFTY FIN SERVICE", "NSE")}

if held_symbols:
    token_lookup = build_nse_token_lookup(kite)
    for i, symbol in enumerate(held_symbols, start=1):
        try:
            if symbol in index_symbols:
                tradingsymbol, exchange = index_symbols[symbol]
                token = get_index_token(kite, tradingsymbol, exchange)
            else:
                token = token_lookup.get(symbol)

            if token is None:
                print(f"[{i}/{len(held_symbols)}] {symbol} — no token found, skipping")
                continue

            process_stock_incremental(symbol, token, kite, to_date=datetime.now())
            print(f"[{i}/{len(held_symbols)}] {symbol} — refreshed")
        except Exception as e:
            print(f"[{i}/{len(held_symbols)}] {symbol} — FAILED: {e}")
else:
    print("Nothing to refresh.")

Fetching 2022-01-01 to 2023-01-31...
Fetching 2023-02-01 to 2024-03-02...
Fetching 2024-03-03 to 2025-04-02...
Fetching 2025-04-03 to 2026-05-03...
Fetching 2026-05-04 to 2026-10-27...
Saved 2376 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/DABUR_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/DABUR_full_report.xlsx
[1/3] DABUR — refreshed
Fetching 2022-01-01 to 2023-01-31...
Fetching 2023-02-01 to 2024-03-02...
Fetching 2024-03-03 to 2025-04-02...
Fetching 2025-04-03 to 2026-05-03...
Fetching 2026-05-04 to 2026-10-27...
Saved 2376 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/PIIND_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/PIIND_full_report.xlsx
[2/3] PIIND — refreshed
Fetching 2022-01-01 to 2023-01-31...
Fetching 2023-02-01 to 2024-03-02...
Fetching 2024-03-03 to 2025-04-02...
Fetching 2025-04-03 to 2026-05-03...
Fetching 2026-05-04 to 2026-10-27...
Saved 2376 candles to /Users/hgmac/Document

In [3]:
# ============================================================================
#  POSITION HEALTH — this is the cell to read.
# ============================================================================
# Groups your option legs by underlying, infers each one's directional thesis,
# and compares it against the freshly-refreshed screener state. Flagged rows
# (FLIP REVERSED / signal weakened / thin buffer) surface first — everything
# is still your call on what to do about it.

from src.position_monitor import build_position_report

report = build_position_report(kite)

if report.empty:
    print("No open option positions to report on.")
else:
    report["_flagged"] = report["thesis_status"].str.startswith("⚠")
    report = report.sort_values(["_flagged", "live_pnl"], ascending=[False, True]).drop(columns="_flagged")

    total_pnl = report["live_pnl"].sum()
    n_flagged = report["thesis_status"].str.startswith("⚠").sum()
    print(f"Total live P&L: {total_pnl:,.2f}   |   {n_flagged}/{len(report)} position(s) flagged")
    print("SORTED BY: flagged first, then live_pnl ascending (biggest drawdown on top)\n")

    display(report[["symbol", "structure", "legs", "short_qty", "long_qty", "live_pnl", "dte",
                     "current_direction", "trend_age_days", "buffer_to_flip_pct",
                     "moved_since_entry_pct", "edge_flag", "thesis_status"]])

    for _, r in report.iterrows():
        print(f"\n{r['symbol']} ({r['structure']}, short {r['short_qty']} / long {r['long_qty']} qty, "
              f"{r['dte']}d to expiry, P&L {r['live_pnl']:,.2f})")
        print(f"   {r['thesis_status']}")

print("\nStill entirely manual — nothing above places, closes, or modifies any order.")

Total live P&L: 1,830.00   |   0/3 position(s) flagged
SORTED BY: flagged first, then live_pnl ascending (biggest drawdown on top)



,symbol,structure,legs,short_qty,long_qty,live_pnl,dte,current_direction,trend_age_days,buffer_to_flip_pct,moved_since_entry_pct,edge_flag,thesis_status
2,TVSMOTOR,BULL PUT,2,175,175,402.5,0,UP,12,4.12,0.31,,thesis intact
1,PIIND,BEAR CALL,2,350,350,490.0,0,DOWN,26,2.40,6.80,,thesis intact
0,DABUR,BEAR CALL,2,3750,3750,937.5,0,DOWN,26,4.23,5.57,,thesis intact



TVSMOTOR (BULL PUT, short 175 / long 175 qty, 0d to expiry, P&L 402.50)
   thesis intact

PIIND (BEAR CALL, short 350 / long 350 qty, 0d to expiry, P&L 490.00)
   thesis intact

DABUR (BEAR CALL, short 3750 / long 3750 qty, 0d to expiry, P&L 937.50)
   thesis intact

Still entirely manual — nothing above places, closes, or modifies any order.
